# 08 · Final pipeline, serialisation and deployment

How the selected model is packaged, loaded and served by the Streamlit app:

* **Reproducibility:** checks against the data hash, seeds, library versions and predictions.
* **Inference:** input validation and single-record prediction.
* **Speed:** timing for single records and batches.
* **Deployment:** how the app consumes the artefacts.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from talentscope import config
from talentscope.reporting import figures as F
F.setup()
print("TalentScope root:", ROOT.name, "| seed", config.RANDOM_SEED)

TalentScope root: TalentScope | seed 42


## 1. The artefacts on disk

In [2]:
import json, joblib
from talentscope.modeling import load_bundle
b = load_bundle()
for p in sorted((config.DEFAULT_EXPERIMENT_DIR).rglob("*")) + sorted(config.DEFAULT_MODEL_DIR.glob("*")):
    if p.is_file():
        print(f"{p.relative_to(config.ROOT).as_posix():55s} {p.stat().st_size / 1024:8.1f} KB")

experiments/default/best_params.json                         3.5 KB
experiments/default/calibration_oof.csv                      0.2 KB
experiments/default/calibration_test.csv                     0.1 KB
experiments/default/classification_report.csv                0.2 KB
experiments/default/config.json                              0.5 KB
experiments/default/cv_folds.csv                            39.3 KB
experiments/default/cv_summary.csv                           7.7 KB
experiments/default/explain/explain_metadata.json            0.3 KB
experiments/default/explain/group_metrics.csv                1.6 KB
experiments/default/explain/permutation_importance.csv       2.2 KB
experiments/default/explain/shap_importance.csv              4.2 KB
experiments/default/explain/shap_importance_by_feature.csv      1.1 KB
experiments/default/explain/shap_test.npz                   33.1 KB
experiments/default/metadata.json                            2.9 KB
experiments/default/oof_predictions.csv      

In [3]:
pd.Series({"selected": b.label, "params": b.selection["params"], "threshold": b.threshold,
           "created": b.metadata["created_utc"], "runtime (s)": b.metadata["runtime_seconds"]}).to_frame("value")

,value
selected,Logistic regression + No rebalancing
params,{'C': 0.2537815508265665}
threshold,0.36
created,2026-11-09T14:22:30Z
runtime (s),606.7


## 2. Reproducibility checks

In [4]:
from talentscope.data import dataset_fingerprint, load_dataset, split_features_target
from talentscope.preprocessing import make_split
import sklearn, imblearn, xgboost, lightgbm, shap
checks = {
    "dataset sha256 matches the run": dataset_fingerprint() == b.metadata["dataset"]["sha256"],
    "seed recorded": b.config["seed"] == config.RANDOM_SEED,
    "scikit-learn version matches": sklearn.__version__ == b.metadata["versions"]["scikit-learn"],
    "imbalanced-learn version matches": imblearn.__version__ == b.metadata["versions"]["imbalanced-learn"],
}
model = joblib.load(config.DEFAULT_MODEL_DIR / "model.joblib")
df = load_dataset(); X, y = split_features_target(df)
X_train, X_test, y_train, y_test = make_split(X, y, b.config["test_size"], b.config["seed"])
p = model.predict_proba(X_test)[:, 1]
checks["test row ids match saved predictions"] = list(X_test.index) == list(b.test_predictions["row_id"])
checks["loaded model reproduces saved probabilities"] = bool(np.allclose(p, b.test_predictions["probability"], atol=1e-6))
pd.Series(checks).to_frame("ok")

,ok
dataset sha256 matches the run,True
seed recorded,True
scikit-learn version matches,True
imbalanced-learn version matches,True
test row ids match saved predictions,True
loaded model reproduces saved probabilities,True


## 3. Round-trip serialisation

A model saved with `joblib` and reloaded must give identical outputs. The pipeline includes the imputer, the encoder
and the threshold-free model, so a single file is enough to serve raw records.

In [5]:
import tempfile, os
with tempfile.TemporaryDirectory() as d:
    path = os.path.join(d, "m.joblib"); joblib.dump(model, path, compress=3)
    again = joblib.load(path)
    print("identical after round trip:", np.array_equal(again.predict_proba(X_test), model.predict_proba(X_test)),
          f"| file size {os.path.getsize(path) / 1024:.1f} KB")
model

identical after round trip: True | file size 3.9 KB


,steps,"[('impute', ...), ('encode', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
Name,Type,Value
classes_,"ndarray[int64](2,)","[0,1]"
feature_names_in_,"ndarray[object](30,)","['Age','DailyRate','DistanceFromHome',...,'JobRole','MaritalStatus', 'OverTime']"
n_features_in_,int,30
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature_name}__{transformer_name}""``. See :meth:`str.format` method from the standard library for more info... versionadded:: 1.0.. versionchanged:: 1.6 `verbose_feature_names_out` can be a callable or a string to be formatted.",False
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'


## 4. Inference on a single record, with validation

In [6]:
from talentscope.modeling.inference import typical_employee, validate_employee, predict_employee
rec = typical_employee(X_train)
res = predict_employee(model, rec, b.threshold)
print(f"typical employee: {res.probability:.1%} -> {res.band} (low < {res.low_cut:.0%} <= elevated < {res.threshold:.0%} <= high)")
risky = dict(rec, OverTime="Yes", JobSatisfaction=1, BusinessTravel="Travel_Frequently", YearsSinceLastPromotion=4,
             NumCompaniesWorked=6, MaritalStatus="Single")
r2 = predict_employee(model, risky, b.threshold)
print(f"same employee with several risk factors: {r2.probability:.1%} -> {r2.band}")

typical employee: 1.4% -> low (low < 18% <= elevated < 36% <= high)
same employee with several risk factors: 65.1% -> high


In [7]:
bad = dict(rec, Age=22, TotalWorkingYears=15, YearsAtCompany=18, OverTime="Sometimes")
for e in validate_employee(bad):
    print("✗", e)
try:
    predict_employee(model, bad, b.threshold)
except ValueError as exc:
    print("\nprediction refused:", type(exc).__name__)

✗ Overtime must be one of ['No', 'Yes'].
✗ Years at company cannot exceed total working years.
✗ Total working years is implausible for this age (career would start before 14).

prediction refused: ValueError


## 5. Latency

In [8]:
import time
from talentscope.data import schema
from talentscope.modeling.inference import to_frame
one = to_frame(rec)
t = time.perf_counter()
for _ in range(200):
    model.predict_proba(one)
single = (time.perf_counter() - t) / 200
t = time.perf_counter(); model.predict_proba(X[schema.MODEL_FEATURES]); batch = time.perf_counter() - t
print(f"single record: {single * 1000:.2f} ms | all 1,470 employees: {batch * 1000:.1f} ms")

single record: 13.24 ms | all 1,470 employees: 18.9 ms


## 6. How the Streamlit app uses these artefacts

```
app.py                          st.navigation, brand bar, navigation rail
src/talentscope/ui/state.py     @st.cache_data     dataset(), split_frames()
                                @st.cache_resource load_bundle(run_dir), joblib model, SHAP explainer
app_pages/6_model_performance   reads cv_summary.csv, cv_folds.csv, oof_predictions.csv, test_*.{csv,json}
app_pages/7_explainable_ai      reads explain/shap_test.npz, permutation_importance.csv
app_pages/8_risk_simulator      model.joblib + validate_employee + LinearExplainer for the local waterfall
app_pages/5_model_laboratory    launches talentscope.modeling.run_job in a separate process -> experiments/runs/<id>
```

* No metric is typed into the UI. Every number on the model pages is read from these files on load.
* If no model exists, the pages show an empty state that points to `python scripts/train_default.py`.
* Lab runs are written next to the default run and appear in every page's experiment picker.

## 7. The model card

In [9]:
from talentscope.reporting import model_card
from talentscope.explainability import load_explanations
from IPython.display import Markdown
Markdown(model_card(b, load_explanations(b)))

# Model card: TalentScope attrition model (Logistic regression, no rebalancing)

Generated 2026-11-09 15:08 UTC from `experiments/default`.

## Model details
- Pipeline: median/mode imputation → integer rounding, standardisation → one-hot encoding → Logistic regression
- Tuned hyperparameters: `{'C': 0.2537815508265665}`
- Decision threshold: 0.36 (out-of-fold predictions on the training set (repeated stratified CV))
- Random seed: 42; libraries: python 3.13.11, scikit-learn 1.9.1, imbalanced-learn 0.14.2, pandas 3.0.6, numpy 2.5.3, xgboost 3.1.3, lightgbm 4.6.0, shap 0.54.1
- Artefact: `models/default/model.joblib` (joblib)

## Intended use
- Portfolio demonstration of leakage-safe imbalanced classification and explainability on a public synthetic dataset.
- Not intended for decisions about real employees: hiring, promotion, pay, performance management or dismissal.

## Training data
- IBM HR Analytics Employee Attrition & Performance, 1,470 rows, sha256 `d11789e1db393cd1d985ca41a0e73a1d405543fb2c0d540a3b4f7d723bca92f7`.
- Source: https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset (copy taken from https://raw.githubusercontent.com/IBM/employee-attrition-aif360/master/data/emp_attrition.csv).
- Fictional data created by IBM data scientists. Identifier and constant columns are removed; 30 attributes are used.
- Stratified split: 1176 training / 294 held-out employees.

## Evaluation (held-out set, scored once, 95% stratified bootstrap intervals)
- PR-AUC: 0.598 (0.477 to 0.713)
- ROC-AUC: 0.814 (0.730 to 0.882)
- F1: 0.530 (0.405 to 0.645)
- Precision: 0.611 (0.472 to 0.758)
- Recall: 0.468 (0.340 to 0.617)
- Balanced accuracy: 0.706 (0.635 to 0.776)
- Brier score: 0.097 (0.082 to 0.114)
- Cross-validated PR-AUC on the training split: 0.662 ± 0.060 (SD over folds).
- Base rate (chance PR-AUC): 0.160.

## Explainability
- Most influential attributes by mean |SHAP|: Overtime, Environment satisfaction, Companies worked, Years since promotion, Job role.
- SHAP values describe the model's use of each input, not causal effects.

## Ethical considerations
- Inputs include gender, age and marital status. Error rates by group are reported in the evaluation summary; differences there come from small groups and from patterns in synthetic data.
- A risk score must not be used to treat employees differently. If used at all, it should prompt a supportive conversation reviewed by a person.

## Caveats
- Small sample: 237 leavers in total. Expect the held-out metrics to move by several points on another split.
- The data is synthetic, so the patterns may not transfer to any real organisation.


## Summary

* One `joblib` file holds the full preprocessing and model pipeline; the threshold and metadata sit beside it in JSON.
* Reloading reproduces the saved test probabilities exactly, and the data hash and library versions match the run.
* Invalid records are rejected with readable reasons before scoring.
* Single-record inference takes on the order of 10 to 20 ms (mostly pandas overhead), fast enough for the interactive
  simulator; scoring all 1,470 employees takes about the same time.